# Node2Vec Graph Embeddings

Train Node2Vec on the co-occurrence graph using PecanPy (C-optimized).

**Input**: TSV `termA\ttermB\tweight` (uploaded as Kaggle Dataset)
**Output**: `embeddings.bin` (for Go) + `embeddings.npz` (for Python)

In [ ]:
!pip install -q pecanpy networkx numpy

In [ ]:
import os

OUTPUT_DIR = "/kaggle/working/models/embeddings_v1"
GRAPH_TSV = "/kaggle/input/cooccurrence-graph/graph.tsv"
DIM = 128
WALKS = 200
WALK_LENGTH = 30
P = 1
Q = 0.5
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
import numpy as np
import networkx as nx

# Load edge list (skip header)
G = nx.Graph()
with open(GRAPH_TSV) as f:
    next(f)  # skip header
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) >= 3:
            G.add_edge(parts[0], parts[1], weight=float(parts[2]))

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

In [ ]:
from pecanpy import pecanpy

# Write edge list in PecanPy format (space-separated, no header)
edgelist_path = os.path.join(OUTPUT_DIR, "edges.edg")
with open(edgelist_path, "w") as f:
    for u, v, d in G.edges(data=True):
        f.write(f"{u} {v} {d['weight']:.6f}\n")

# Train Node2Vec
g = pecanpy.SparseOTF(p=P, q=Q, workers=4, verbose=True)
g.read_edg(edgelist_path, weighted=True, directed=False)
embeddings = g.embed(dim=DIM, num_walks=WALKS, walk_length=WALK_LENGTH)

# Get node IDs in order
node_ids = g.nodes
print(f"Trained embeddings: {embeddings.shape}")

In [ ]:
import struct

# Export binary format for Go:
# [count:u32][dim:u32][entries: termLen:u16|term|vector:dim×f32]
bin_path = os.path.join(OUTPUT_DIR, "embeddings.bin")
with open(bin_path, "wb") as f:
    f.write(struct.pack("<II", len(node_ids), DIM))
    for i, node_id in enumerate(node_ids):
        term_bytes = node_id.encode("utf-8")
        f.write(struct.pack("<H", len(term_bytes)))
        f.write(term_bytes)
        f.write(struct.pack(f"<{DIM}f", *embeddings[i]))

print(f"Binary: {os.path.getsize(bin_path) / 1024 / 1024:.1f} MB")

# Also save as NPZ for Python analysis
npz_path = os.path.join(OUTPUT_DIR, "embeddings.npz")
np.savez(npz_path, embeddings=embeddings, node_ids=np.array(node_ids))
print(f"NPZ: {os.path.getsize(npz_path) / 1024 / 1024:.1f} MB")

In [ ]:
# Validate — sample cosine similarities
from numpy.linalg import norm

def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

# Pick some connected nodes and verify high similarity
sample_edges = list(G.edges())[:5]
id_to_idx = {nid: i for i, nid in enumerate(node_ids)}

for u, v in sample_edges:
    if u in id_to_idx and v in id_to_idx:
        sim = cosine_sim(embeddings[id_to_idx[u]], embeddings[id_to_idx[v]])
        print(f"  {u} <-> {v}: cosine={sim:.3f} (edge weight={G[u][v]['weight']:.3f})")